<a href="https://colab.research.google.com/github/gracenaomi1122/my-first-repo/blob/main/Assignment_Model_Validation_and_Data_Issues_Subjective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Step 1: Build the DataFrame from the given sample data
data = {
    "V1": [0.12, -0.33, 0.67, -1.02, 0.44, -0.28, 0.91, -0.19, 0.05, -2.75, -3.02, -2.91],
    "V2": [-0.45, 0.88, 0.21, -0.15, 0.09, 0.55, -0.62, 0.31, -0.88, 3.10, 2.88, 3.45],
    "Amount": [25.50, 120.00, 45.75, 300.10, 15.20, 89.99, 5.00, 210.40, 60.00, 2.50, 400.00, 1.20],
    "Class": [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]
}

df = pd.DataFrame(data)

# Step 2: Manual class-weight formula
def compute_class_weight(n_samples, n_classes, n_samples_in_class):
    return n_samples / (n_classes * n_samples_in_class)

# Step 3: Compute class weights
n_samples = len(df)
n_classes = df["Class"].nunique()

class_counts = df["Class"].value_counts().sort_index()

weights = {}
for cls, count in class_counts.items():
    weights[cls] = compute_class_weight(n_samples, n_classes, count)

print("Manual Class Weights:")
for cls, weight in weights.items():
    print(f"Class {cls}: {weight:.3f}")

# Features and target
X = df[["V1", "V2", "Amount"]]
y = df["Class"]

# Step 4: Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print("\nTraining Class Distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting Class Distribution:")
print(y_test.value_counts().sort_index())

# Step 5: Feature Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 6: Train Logistic Regression models

# Baseline model
baseline_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

baseline_model.fit(X_train_scaled, y_train)

# Balanced model
balanced_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

balanced_model.fit(X_train_scaled, y_train)

# Step 7: Evaluation Function
def evaluate_model(name, model):
    y_pred = model.predict(X_test_scaled)

    print(f"\n{name}")
    print("-" * len(name))
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, pos_label=1, zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred, pos_label=1, zero_division=0):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_pred, pos_label=1, zero_division=0):.4f}")

# Evaluate both models
evaluate_model("Baseline Logistic Regression", baseline_model)
evaluate_model("Balanced Logistic Regression", balanced_model)

if __name__ == "__main__":
    pass

Manual Class Weights:
Class 0: 0.667
Class 1: 2.000

Training Class Distribution:
Class
0    7
1    2
Name: count, dtype: int64

Testing Class Distribution:
Class
0    2
1    1
Name: count, dtype: int64

Baseline Logistic Regression
----------------------------
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000

Balanced Logistic Regression
----------------------------
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
